# STARmap PLUS preprocessing

Prepare the eight STARmap PLUS samples used by Popari and generate orientation-corrected pathology images for downstream analysis.

In [1]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from PIL import Image
from scipy.sparse import csr_matrix

from popari import pl, pp, tl
from popari.io import load_anndata, save_anndata

## Configuration

In [2]:
data_directory = Path(
    "/work/magroup/shahula/spatiotemporal_transcriptomics_integration/data/STARmapPlus/SCP1375"
)
expression_path = data_directory / "expression" / "expression_matrix_raw.csv"
output_path = data_directory / "eight_replicate_hvgs_None.h5ad"
corrected_image_directory = data_directory / "derived" / "orientation_corrected_images"
source_image_directory = (
    data_directory
    / "documentation"
    / "2022-11-17-mAD-2766-genes-protein-images"
)

SAMPLE_SPECS = {
    "8mon_dis_repl1": {
        "annotations": "spatial_8months-disease-replicate_1.csv",
        "plaque_metadata": "plaque_8months-disease-replicate_1.csv",
        "image_directory": "8months_disease-replicate_1",
        "image_flips": {"plaque": (0,), "tau": (0, 1)},
    },
    "8mon_dis_repl2": {
        "annotations": "spatial_8months-disease-replicate_2.csv",
        "plaque_metadata": "plaque_8months-disease-replicate_2.csv",
        "image_directory": "8months_disease-replicate_2",
        "image_flips": {"plaque": (1,), "tau": ()},
    },
    "8mon_contr_repl1": {
        "annotations": "spatial_8months-control-replicate_1.csv",
    },
    "8mon_contr_repl2": {
        "annotations": "spatial_8months-control-replicate_2.csv",
    },
    "13mon_dis_repl1": {
        "annotations": "spatial_13months-disease-replicate_1.csv",
        "plaque_metadata": "plaque_13months-disease-replicate_1.csv",
        "image_directory": "13months_disease-replicate_1",
        "image_flips": {"plaque": (), "tau": ()},
    },
    "13mon_dis_repl2": {
        "annotations": "spatial_13months-disease-replicate_2.csv",
        "plaque_metadata": "plaque_13months-disease-replicate_2.csv",
        "image_directory": "13months_disease-replicate_2",
        "image_flips": {"plaque": (), "tau": ()},
    },
    "13mon_contr_repl1": {
        "annotations": "spatial_13months-control-replicate_1.csv",
    },
    "13mon_contr_repl2": {
        "annotations": "spatial_13months-control-replicate_2.csv",
    },
}

## Prepare orientation-corrected pathology images

These files retain the source pixels and TIFF mode. The only transformations are the sample-specific axis flips listed in `SAMPLE_SPECS`.

In [3]:
def save_tiff(image, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.ascontiguousarray(image)).save(output_path)
    return output_path


def orient_image(source_path, flip_axes=()):
    if not source_path.is_file():
        raise FileNotFoundError(f"Source image does not exist: {source_path}")

    with Image.open(source_path) as source_image:
        image = np.asarray(source_image)

    for axis in flip_axes:
        image = np.flip(image, axis=axis)
    return np.ascontiguousarray(image)


def adjust_contrast(image, c0=10, c1=0.3, eps=1e-5, percentile=100):
    """Apply the sigmoid contrast transform used in STARmap+ figures."""

    scaled_image = image / (np.percentile(image, percentile) + eps)
    return 1 / (1 + np.exp(-(scaled_image - c1) * c0))


def normalize_to_uint8(image):
    """Min-max normalize an image onto the unsigned 8-bit intensity range."""

    image = image.astype(np.float64)
    intensity_range = np.ptp(image)
    if intensity_range == 0:
        return np.zeros_like(image, dtype=np.uint8)
    return (np.iinfo(np.uint8).max * (image - image.min()) / intensity_range).astype(np.uint8)


corrected_image_paths = {}
for sample_name, sample_spec in SAMPLE_SPECS.items():
    if "image_directory" not in sample_spec:
        continue

    corrected_image_paths[sample_name] = {}
    sample_image_directory = source_image_directory / sample_spec["image_directory"]
    source_filenames = {"plaque": "plaque.tif", "tau": "tau_mask.tif"}
    for image_key, source_filename in source_filenames.items():
        oriented_image = orient_image(
            sample_image_directory / source_filename,
            flip_axes=sample_spec["image_flips"][image_key],
        )

        oriented_path = corrected_image_directory / f"{sample_name}_{image_key}.tif"
        adjusted_key = f"{image_key}_contrast_adjusted"
        adjusted_path = corrected_image_directory / f"{sample_name}_{adjusted_key}.tif"
        save_tiff(oriented_image, oriented_path)
        save_tiff(normalize_to_uint8(adjust_contrast(oriented_image)), adjusted_path)

        with Image.open(oriented_path) as saved_oriented:
            np.testing.assert_array_equal(np.asarray(saved_oriented), oriented_image)
        with Image.open(adjusted_path) as saved_adjusted:
            if saved_adjusted.mode != "L" or saved_adjusted.size != (oriented_image.shape[1], oriented_image.shape[0]):
                raise ValueError(f"Unexpected adjusted image metadata: {adjusted_path}")

        corrected_image_paths[sample_name][image_key] = oriented_path
        corrected_image_paths[sample_name][adjusted_key] = adjusted_path

In [4]:
corrected_image_paths

{'8mon_dis_repl1': {'plaque': PosixPath('/work/magroup/shahula/spatiotemporal_transcriptomics_integration/data/STARmapPlus/SCP1375/derived/orientation_corrected_images/8mon_dis_repl1_plaque.tif'),
  'plaque_contrast_adjusted': PosixPath('/work/magroup/shahula/spatiotemporal_transcriptomics_integration/data/STARmapPlus/SCP1375/derived/orientation_corrected_images/8mon_dis_repl1_plaque_contrast_adjusted.tif'),
  'tau': PosixPath('/work/magroup/shahula/spatiotemporal_transcriptomics_integration/data/STARmapPlus/SCP1375/derived/orientation_corrected_images/8mon_dis_repl1_tau.tif'),
  'tau_contrast_adjusted': PosixPath('/work/magroup/shahula/spatiotemporal_transcriptomics_integration/data/STARmapPlus/SCP1375/derived/orientation_corrected_images/8mon_dis_repl1_tau_contrast_adjusted.tif')},
 '8mon_dis_repl2': {'plaque': PosixPath('/work/magroup/shahula/spatiotemporal_transcriptomics_integration/data/STARmapPlus/SCP1375/derived/orientation_corrected_images/8mon_dis_repl2_plaque.tif'),
  'plaqu

## Construct the unified AnnData

In [5]:
raw_expression = pd.read_csv(expression_path, index_col="GENE")
raw_expression.columns = raw_expression.columns.astype(str)
raw_expression.index = raw_expression.index.astype(str)

if not raw_expression.index.is_unique:
    raise ValueError("Gene names in the raw expression matrix must be unique.")
if not raw_expression.columns.is_unique:
    raise ValueError("Cell IDs in the raw expression matrix must be unique.")

In [6]:
REQUIRED_ANNOTATION_COLUMNS = {
    "NAME",
    "X-scaled",
    "Y-scaled",
    "sub_level_cell_type",
    "top_level_cell_type",
}


def read_starmap_table(path):
    if not path.is_file():
        raise FileNotFoundError(f"STARmap PLUS input does not exist: {path}")
    return pd.read_csv(path, header=0, skiprows=lambda row: row == 1)


def build_sample(sample_name, sample_spec):
    annotations = read_starmap_table(
        data_directory / "cluster" / sample_spec["annotations"]
    )
    missing_columns = REQUIRED_ANNOTATION_COLUMNS.difference(annotations.columns)
    if missing_columns:
        raise ValueError(
            f"{sample_name}: annotation table is missing columns {sorted(missing_columns)}."
        )

    cell_ids = annotations["NAME"].astype(str)
    if cell_ids.duplicated().any():
        raise ValueError(f"{sample_name}: annotation cell IDs must be unique.")
    missing_cells = cell_ids[~cell_ids.isin(raw_expression.columns)]
    if not missing_cells.empty:
        raise ValueError(
            f"{sample_name}: {len(missing_cells)} annotated cells are absent from the expression matrix."
        )

    expression = raw_expression.loc[:, cell_ids].T
    obs = annotations[["sub_level_cell_type", "top_level_cell_type"]].copy()
    obs["cell_id"] = cell_ids.to_numpy()
    obs.index = pd.Index(
        [f"{sample_name}:{cell_id}" for cell_id in cell_ids],
        name="observation",
    )

    sample = ad.AnnData(
        X=csr_matrix(expression.to_numpy(dtype=np.float32)),
        obs=obs,
    )
    sample.var_names = raw_expression.index.copy()
    sample.obsm["spatial"] = annotations[["X-scaled", "Y-scaled"]].to_numpy()
    return sample


sample_datasets = {
    sample_name: build_sample(sample_name, sample_spec)
    for sample_name, sample_spec in SAMPLE_SPECS.items()
}
adata = ad.concat(
    sample_datasets,
    label="batch",
    index_unique=None,
    join="inner",
)
adata.obs["batch"] = pd.Categorical(
    adata.obs["batch"].astype(str),
    categories=list(SAMPLE_SPECS),
    ordered=True,
)
adata.popari.name = "STARmapPlus"

## Normalize expression and construct spatial graphs

In [7]:
# Remove empty cells before deriving a shared gene set.
sc.pp.filter_cells(adata, min_counts=1)

# Keep genes observed in at least three cells in every sample.
valid_genes = np.ones(adata.n_vars, dtype=bool)
for sample_name in adata.popari.sample_names:
    sample = adata[adata.popari.sample_indices(sample_name)]
    sample_valid_genes, _ = sc.pp.filter_genes(
        sample,
        min_cells=3,
        inplace=False,
    )
    valid_genes &= sample_valid_genes
adata = adata[:, valid_genes].copy()

# Match normalize_total's joint-dataset default using one global median library size.
cell_totals = np.asarray(adata.X.sum(axis=1)).ravel()
target_sum = float(np.median(cell_totals[cell_totals > 0]))
sc.pp.normalize_total(adata, target_sum=target_sum)
sc.pp.log1p(adata)

pp.compute_spatial_neighbors(adata)

INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        


/work/magroup/shahula/spatiotemporal_transcriptomics_integration/Popari/.venv/lib/python3.12/site-packages/anndata/utils.py:362: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)


INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
INFO     Creating graph using `generic` coordinates and `None` transform

(AnnData object with n_obs × n_vars = 8186 × 2766
     obs: 'sub_level_cell_type', 'top_level_cell_type', 'cell_id', 'n_counts'
     uns: 'dataset_name', 'log1p', 'spatial_neighbors'
     obsm: 'spatial', 'adjacency_list'
     obsp: 'spatial_connectivities', 'spatial_distances', 'adjacency_matrix',
 AnnData object with n_obs × n_vars = 8202 × 2766
     obs: 'sub_level_cell_type', 'top_level_cell_type', 'cell_id', 'n_counts'
     uns: 'dataset_name', 'log1p', 'spatial_neighbors'
     obsm: 'spatial', 'adjacency_list'
     obsp: 'spatial_connectivities', 'spatial_distances', 'adjacency_matrix',
 AnnData object with n_obs × n_vars = 8506 × 2766
     obs: 'sub_level_cell_type', 'top_level_cell_type', 'cell_id', 'n_counts'
     uns: 'dataset_name', 'log1p', 'spatial_neighbors'
     obsm: 'spatial', 'adjacency_list'
     obsp: 'spatial_connectivities', 'spatial_distances', 'adjacency_matrix',
 AnnData object with n_obs × n_vars = 9803 × 2766
     obs: 'sub_level_cell_type', 'top_level_cell_t

## Save and validate the production dataset

In [8]:
output_path.parent.mkdir(parents=True, exist_ok=True)
save_anndata(output_path, adata)

loaded_adata = load_anndata(output_path)
expected_names = tuple(SAMPLE_SPECS)
if loaded_adata.popari.sample_names != expected_names:
    raise ValueError(
        f"Expected sample order {expected_names}, "
        f"found {loaded_adata.popari.sample_names}."
    )
if loaded_adata.n_obs != 72_165:
    raise ValueError("The saved dataset does not contain the expected 72,165 cells.")
if not loaded_adata.obs_names.is_unique:
    raise ValueError("Observation names are not unique.")
if not np.isfinite(loaded_adata.X.data).all() or (loaded_adata.X.data < 0).any():
    raise ValueError("Expression contains invalid values.")
if "adjacency_matrix" not in loaded_adata.obsp:
    raise ValueError("The block-diagonal spatial adjacency matrix is missing.")

{
    "output": output_path,
    "samples": len(loaded_adata.popari.sample_names),
    "cells": loaded_adata.n_obs,
    "genes": loaded_adata.n_vars,
}

/work/magroup/shahula/spatiotemporal_transcriptomics_integration/Popari/.venv/lib/python3.12/site-packages/anndata/utils.py:362: ExperimentalFeatureWarning: Outer joins on awkward.Arrays will have different return values in the future. For details, and to offer input, please see:

	https://github.com/scverse/anndata/issues/898
  warnings.warn(msg, category, stacklevel=stacklevel)


{'output': PosixPath('/work/magroup/shahula/spatiotemporal_transcriptomics_integration/data/STARmapPlus/SCP1375/eight_replicate_hvgs_None.h5ad'),
 'samples': 8,
 'cells': 72165,
 'genes': 2766}

## Batch-effect diagnostic

This visualization is diagnostic and does not alter the saved preprocessing artifact.

In [ ]:
umap_dataset = loaded_adata.copy()

In [ ]:
pp.pca(umap_dataset, n_comps=50)
tl.umap(umap_dataset, use_rep="X_pca")

In [ ]:
umap_figure = sc.pl.umap(
    umap_dataset,
    color="batch",
    edges=False,
    size=2,
    return_fig=True,
)